# Phase 4: Mixup Barcode + Ordinary PI 연결 방식 개선

## 목적
Mixup Barcode 기반 벡터(Inter_PI, 3D_PI)는 정보량이 많음에도 단순 concatenation 시
Ordinary PI보다 성능이 낮음. 이를 개선하기 위한 다양한 결합 전략을 실험.

## 실험 설계
| # | 방법 | 설명 |
|---|------|------|
| B | Baseline (Raw Concat) | 기존 방식: X = [Inter_PI \| Ord_PI] |
| N | Individual Normalization | 각각 StandardScaler → concat |
| P | Independent PCA → Concat | 각각 PCA 축소 후 concat |
| W | Weighted Concat | λ * Inter_PI + (1-λ) * Ord_PI (λ 탐색) |
| NP| Norm + PCA | 정규화 후 각각 PCA → concat |

## 평가 파이프라인 (최종.ipynb 기준)
- StandardScaler → PCA 20D → 5-fold StratifiedKFold
- Soft Accuracy (ADJACENT_PHASES: phase diagram 추출, 27쌍)
- 분류기: SVM (RBF/Linear), KNN (k=3, 12), Random Forest

## 1. 환경 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, glob, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.base import clone
import warnings
warnings.filterwarnings('ignore')
matplotlib.rcParams['axes.unicode_minus'] = False

print('All imports loaded.')

## 2. Ground Truth & Adjacent Phases

In [ ]:
M1=[[0,0,1,1,1,1,1,1],[0,0,1,1,1,1,1,1],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3]]
M2=[[0,0,1,1,1,1,1,1],[0,0,1,1,1,1,1,1],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,4],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,4,4],[2,2,3,3,3,3,3,3]]
M3=[[6,6,7,7,7,7,7,7],[6,6,6,7,7,7,7,7],[9,6,3,3,3,3,3,3],[9,10,3,4,4,3,3,4],[9,10,3,3,4,4,3,4],[9,10,3,4,4,4,4,4],[9,10,3,4,3,4,4,4],[9,10,3,4,3,4,4,4]]
M4=[[6,6,12,12,7,7,7,7],[6,6,12,12,7,7,7,7],[9,6,6,11,7,7,4,4],[9,9,6,3,3,4,4,4],[9,9,10,3,3,4,4,4],[9,9,10,3,3,4,4,4],[9,9,10,4,4,4,4,4],[9,9,10,4,4,4,4,4]]
M5=[[6,6,12,12,12,12,7,7],[6,6,12,12,12,12,12,7],[9,9,6,11,11,11,12,11],[9,9,6,11,11,11,4,4],[9,9,13,13,4,4,4,4],[9,9,13,10,4,4,4,4],[9,9,13,10,4,4,4,4],[9,9,10,10,4,4,4,4]]
M6=[[6,12,12,12,12,12,12,12],[6,6,12,12,12,12,12,12],[9,6,6,11,11,11,11,11],[9,9,6,11,11,11,11,11],[9,9,6,6,6,13,4,4],[9,9,6,13,13,4,4,4],[9,9,6,13,4,4,4,4],[9,9,6,13,4,4,4,4]]
M7=[[6,6,12,12,12,12,12,12],[9,6,12,12,12,12,12,12],[9,6,6,11,11,11,11,12],[9,6,6,11,11,11,11,11],[9,9,6,6,11,11,11,11],[9,9,6,6,11,11,11,4],[9,9,6,6,13,13,4,4],[9,9,6,13,13,4,4,4]]
M8=[[6,12,12,12,12,12,12,12],[6,6,12,12,12,12,12,12],[9,6,6,6,11,11,11,11],[9,6,6,6,11,11,11,11],[9,9,6,6,11,11,11,11],[9,9,6,6,6,11,11,11],[9,9,6,6,13,13,11,11],[9,9,6,6,13,13,11,4]]
GROUND_TRUTH_M = np.asarray([M1,M2,M3,M4,M5,M6,M7,M8])

def get_label_from_index(task_id):
    idx = task_id - 1
    RR_idx = idx // 64
    RG_idx = (idx % 64) // 8
    GG_idx = idx % 8
    return GROUND_TRUTH_M[RG_idx][RR_idx][GG_idx]

# Phase diagram에서 직접 추출 (최종.ipynb final code 기준, 27쌍)
def extract_adjacent_phases(matrices):
    adjacent_pairs = set()
    for M in matrices:
        M = np.array(M)
        rows, cols = M.shape
        for i in range(rows):
            for j in range(cols):
                current = M[i, j]
                for ni, nj in [(i-1,j),(i+1,j),(i,j-1),(i,j+1)]:
                    if 0 <= ni < rows and 0 <= nj < cols and M[ni,nj] != current:
                        adjacent_pairs.add(tuple(sorted([int(current), int(M[ni,nj])])))
    adj_dict = {}
    for p1, p2 in adjacent_pairs:
        adj_dict.setdefault(p1, []).append(p2)
        adj_dict.setdefault(p2, []).append(p1)
    return adj_dict

ADJACENT_PHASES = extract_adjacent_phases(GROUND_TRUTH_M)

def are_adjacent_phases(label1, label2, adj=None):
    if adj is None: adj = ADJACENT_PHASES
    l1, l2 = int(label1), int(label2)
    if l1 == l2: return True
    return (l1 in adj and l2 in adj[l1]) or (l2 in adj and l1 in adj[l2])

def soft_accuracy_score(y_true, y_pred):
    n = len(y_true)
    correct = sum(1 for t, p in zip(y_true, y_pred)
                  if are_adjacent_phases(t, p))
    return correct / n if n > 0 else 0.0

print(f'Classes: {np.unique(GROUND_TRUTH_M)} ({len(np.unique(GROUND_TRUTH_M))} classes)')
print(f'ADJACENT_PHASES: {len(ADJACENT_PHASES)} entries, '
      f'{sum(len(v) for v in ADJACENT_PHASES.values())//2} pairs')

## 3. 데이터 로딩

In [ ]:
VECTOR_DIR = '/content/drive/MyDrive/URP/1224_Vectors'

DATA_PATHS = {
    'Inter_PI': os.path.join(VECTOR_DIR, 'Inter_PI'),
    'Ord_PI':   os.path.join(VECTOR_DIR, 'Ord_PI'),
    '3D_PI':    os.path.join(VECTOR_DIR, '3D_PI'),
}

def load_generic_pi(data_dir, prefix):
    """Inter_PI / Ord_PI / 3D_PI 공용 로더. arr_0/arr_1이 flat dict인 경우."""
    files = sorted(glob.glob(os.path.join(data_dir, f'{prefix}_*.npz')))
    X_list, y_list, idx_list = [], [], []
    for fp in files:
        fname = os.path.basename(fp)
        try:
            sim_idx = int(fname.split('_')[-1].split('.')[0])
            label = get_label_from_index(sim_idx)
            data = np.load(fp, allow_pickle=True)
            features = []
            for key in ('arr_0', 'arr_1'):
                arr = data[key]
                if hasattr(arr, 'item'):
                    arr = arr.item()
                if isinstance(arr, dict):
                    for k in sorted(arr.keys()):
                        val = arr[k]
                        if isinstance(val, dict):
                            for dk in sorted(val.keys()):
                                features.extend(np.array(val[dk]).flatten())
                        else:
                            features.extend(np.array(val).flatten())
                else:
                    features.extend(np.array(arr).flatten())
            X_list.append(features)
            y_list.append(label)
            idx_list.append(sim_idx)
        except Exception as e:
            print(f'  Error: {fname} - {e}')
    X = np.nan_to_num(np.array(X_list, dtype=float), nan=0., posinf=0., neginf=0.)
    return X, np.array(y_list), idx_list

print('Loading datasets...')
raw = {}
for name, path in DATA_PATHS.items():
    prefix = name.replace('_PI','_PI').replace('+','').replace(' ','')
    # prefix mapping
    prefix_map = {'Inter_PI': 'Inter_PI', 'Ord_PI': 'Ord_PI', '3D_PI': '3D_PI'}
    X, y, idx = load_generic_pi(path, prefix_map[name])
    raw[name] = {'X': X, 'y': y, 'indices': idx}
    print(f'  {name}: {X.shape}, classes={len(np.unique(y))}')

print('\n✓ 데이터 로딩 완료')

## 4. 평가 함수

In [ ]:
CLASSIFIERS = {
    'KNN (k=3)':        KNeighborsClassifier(n_neighbors=3),
    'KNN (k=12)':       KNeighborsClassifier(n_neighbors=12),
    'SVM (RBF, C=1)':   SVC(kernel='rbf',    C=1.0, gamma='scale', random_state=42),
    'SVM (Linear, C=1)':SVC(kernel='linear', C=1.0, random_state=42),
    'SVM (RBF, C=0.5)': SVC(kernel='rbf',    C=0.5, gamma='scale', random_state=42),
    'SVM (RBF, C=2)':   SVC(kernel='rbf',    C=2.0, gamma='scale', random_state=42),
    'Random Forest':    RandomForestClassifier(n_estimators=100, random_state=42),
}

def evaluate(X, y, pca_dim=20, n_splits=5):
    """
    StandardScaler → PCA → 5-fold CV로 각 분류기 평가.
    Returns dict: {clf_name: {soft_mean, soft_std, strict_mean, strict_std, f1_mean}}
    """
    scaler = StandardScaler()
    X_sc = scaler.fit_transform(X)

    if pca_dim is not None and X_sc.shape[1] > pca_dim:
        pca = PCA(n_components=pca_dim, random_state=42)
        X_r = pca.fit_transform(X_sc)
    else:
        X_r = X_sc

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    results = {}

    for name, clf_template in CLASSIFIERS.items():
        s_soft, s_strict, s_f1 = [], [], []
        for tr, te in skf.split(X_r, y):
            clf = clone(clf_template)
            clf.fit(X_r[tr], y[tr])
            yp = clf.predict(X_r[te])
            s_soft.append(soft_accuracy_score(y[te], yp))
            s_strict.append(accuracy_score(y[te], yp))
            s_f1.append(f1_score(y[te], yp, average='weighted', zero_division=0))
        results[name] = {
            'soft_mean':   np.mean(s_soft)   * 100,
            'soft_std':    np.std(s_soft)    * 100,
            'strict_mean': np.mean(s_strict) * 100,
            'strict_std':  np.std(s_strict)  * 100,
            'f1_mean':     np.mean(s_f1)     * 100,
        }
    return results

def best(results):
    k = max(results, key=lambda x: results[x]['soft_mean'])
    return k, results[k]

def print_results(label, results, dim_info=''):
    clf, r = best(results)
    print(f'  [{label}] {dim_info}')
    print(f'    Best: {clf} → Soft {r["soft_mean"]:.2f}±{r["soft_std"]:.2f}% | Strict {r["strict_mean"]:.2f}±{r["strict_std"]:.2f}%')
    return r['soft_mean']

PCA_DIM = 20
N_SPLITS = 5
print(f'평가 설정: PCA {PCA_DIM}D, {N_SPLITS}-fold CV')

## 5. 결합 방식 정의 & 데이터 구성

In [ ]:
def combine_datasets(X_a, X_b, method='raw', pca_dim_each=10, weight_a=0.5,
                      scaler_type='standard'):
    """
    두 feature matrix X_a, X_b를 다양한 방식으로 결합.

    Parameters
    ----------
    method : str
        'raw'      - 단순 concatenation (baseline)
        'norm'     - 개별 StandardScaler 후 concat
        'pca'      - 개별 PCA 후 concat
        'norm_pca' - 정규화 후 개별 PCA → concat
        'weighted' - weight_a * X_a + (1-weight_a) * X_b  (같은 차원일 때)
        'weighted_norm' - 개별 정규화 후 weighted sum
    scaler_type : 'standard' | 'minmax' | 'robust'
    pca_dim_each : 각 측의 PCA 목표 차원 (method='pca'/'norm_pca' 에서 사용)
    weight_a : X_a의 weight (method='weighted*' 에서 사용)
    """
    def get_scaler(stype):
        if stype == 'minmax': return MinMaxScaler()
        if stype == 'robust': return RobustScaler()
        return StandardScaler()

    if method == 'raw':
        return np.hstack([X_a, X_b])

    elif method == 'norm':
        Xa_sc = get_scaler(scaler_type).fit_transform(X_a)
        Xb_sc = get_scaler(scaler_type).fit_transform(X_b)
        return np.hstack([Xa_sc, Xb_sc])

    elif method == 'pca':
        da = min(pca_dim_each, X_a.shape[1])
        db = min(pca_dim_each, X_b.shape[1])
        Xa_r = PCA(n_components=da, random_state=42).fit_transform(StandardScaler().fit_transform(X_a))
        Xb_r = PCA(n_components=db, random_state=42).fit_transform(StandardScaler().fit_transform(X_b))
        return np.hstack([Xa_r, Xb_r])

    elif method == 'norm_pca':
        Xa_sc = get_scaler(scaler_type).fit_transform(X_a)
        Xb_sc = get_scaler(scaler_type).fit_transform(X_b)
        da = min(pca_dim_each, Xa_sc.shape[1])
        db = min(pca_dim_each, Xb_sc.shape[1])
        Xa_r = PCA(n_components=da, random_state=42).fit_transform(Xa_sc)
        Xb_r = PCA(n_components=db, random_state=42).fit_transform(Xb_sc)
        return np.hstack([Xa_r, Xb_r])

    elif method in ('weighted', 'weighted_norm'):
        if method == 'weighted_norm':
            X_a = get_scaler(scaler_type).fit_transform(X_a)
            X_b = get_scaler(scaler_type).fit_transform(X_b)
        # 차원이 다르면 각각 PCA로 맞추기
        # n_components 는 min(n_samples-1, min_feat_dim) 으로 제한
        n_samples = X_a.shape[0]
        target_dim = min(X_a.shape[1], X_b.shape[1], n_samples - 1)
        if X_a.shape[1] != X_b.shape[1]:
            X_a = PCA(n_components=target_dim, random_state=42).fit_transform(X_a)
            X_b = PCA(n_components=target_dim, random_state=42).fit_transform(X_b)
        elif X_a.shape[1] > target_dim:
            X_a = PCA(n_components=target_dim, random_state=42).fit_transform(X_a)
            X_b = PCA(n_components=target_dim, random_state=42).fit_transform(X_b)
        return weight_a * X_a + (1 - weight_a) * X_b

    else:
        raise ValueError(f'Unknown method: {method}')

print('combine_datasets() 함수 정의 완료')


## 6. 실험 A: Inter_PI + Ord_PI 결합 방식 비교

현재 최종.ipynb에서는 단순 raw concat만 사용. 다양한 결합 방식의 성능을 비교.

In [ ]:
X_inter = raw['Inter_PI']['X']
X_ord   = raw['Ord_PI']['X']
y       = raw['Inter_PI']['y']   # Inter_PI와 Ord_PI는 동일 샘플 순서

print(f'Inter_PI: {X_inter.shape}')
print(f'Ord_PI:   {X_ord.shape}')
print()

# 실험할 설정 목록
experiments_A = [
    ('A_B: Baseline (raw concat)',             'raw',       {},                       ),
    ('A_N_std: Norm(std) + concat',            'norm',      {'scaler_type':'standard'}),
    ('A_N_mm: Norm(minmax) + concat',          'norm',      {'scaler_type':'minmax'}  ),
    ('A_N_rob: Norm(robust) + concat',         'norm',      {'scaler_type':'robust'}  ),
    ('A_P10: PCA(10+10) → concat',             'pca',       {'pca_dim_each':10}      ),
    ('A_P20: PCA(20+20) → concat',             'pca',       {'pca_dim_each':20}      ),
    ('A_P50: PCA(50+50) → concat',             'pca',       {'pca_dim_each':50}      ),
    ('A_NP10: Norm+PCA(10+10)',                'norm_pca',  {'pca_dim_each':10}      ),
    ('A_NP20: Norm+PCA(20+20)',                'norm_pca',  {'pca_dim_each':20}      ),
    ('A_W03: Weighted(0.3*Inter)',             'weighted_norm', {'weight_a':0.3}      ),
    ('A_W05: Weighted(0.5*Inter)',             'weighted_norm', {'weight_a':0.5}      ),
    ('A_W07: Weighted(0.7*Inter)',             'weighted_norm', {'weight_a':0.7}      ),
]

results_A = {}
print('=' * 70)
print('실험 A: Inter_PI + Ord_PI')
print('=' * 70)

for label, method, kwargs in experiments_A:
    X_combined = combine_datasets(X_inter, X_ord, method=method, **kwargs)
    res = evaluate(X_combined, y, pca_dim=PCA_DIM, n_splits=N_SPLITS)
    results_A[label] = {'results': res, 'dim': X_combined.shape[1]}
    print_results(label, res, dim_info=f'(combined_dim={X_combined.shape[1]})')

print('=' * 70)

## 7. 실험 B: 3D_PI + Ord_PI 결합 방식 비교

3D_PI는 weight=1로 인해 성능이 낮음. 다양한 결합 전략으로 개선 가능 여부 확인.

In [ ]:
X_3d  = raw['3D_PI']['X']
y_3d  = raw['3D_PI']['y']

print(f'3D_PI:  {X_3d.shape}')
print(f'Ord_PI: {X_ord.shape}')
print()

experiments_B = [
    ('B_B: Baseline (raw concat)',             'raw',       {},                       ),
    ('B_N_std: Norm(std) + concat',            'norm',      {'scaler_type':'standard'}),
    ('B_N_mm: Norm(minmax) + concat',          'norm',      {'scaler_type':'minmax'}  ),
    ('B_N_rob: Norm(robust) + concat',         'norm',      {'scaler_type':'robust'}  ),
    ('B_P10: PCA(10+10) → concat',             'pca',       {'pca_dim_each':10}      ),
    ('B_P20: PCA(20+20) → concat',             'pca',       {'pca_dim_each':20}      ),
    ('B_P50: PCA(50+50) → concat',             'pca',       {'pca_dim_each':50}      ),
    ('B_NP10: Norm+PCA(10+10)',                'norm_pca',  {'pca_dim_each':10}      ),
    ('B_NP20: Norm+PCA(20+20)',                'norm_pca',  {'pca_dim_each':20}      ),
    ('B_W03: Weighted(0.3*3D)',                'weighted_norm', {'weight_a':0.3}      ),
    ('B_W05: Weighted(0.5*3D)',                'weighted_norm', {'weight_a':0.5}      ),
    ('B_W07: Weighted(0.7*3D)',                'weighted_norm', {'weight_a':0.7}      ),
]

results_B = {}
print('=' * 70)
print('실험 B: 3D_PI + Ord_PI')
print('=' * 70)

for label, method, kwargs in experiments_B:
    X_combined = combine_datasets(X_3d, X_ord, method=method, **kwargs)
    res = evaluate(X_combined, y_3d, pca_dim=PCA_DIM, n_splits=N_SPLITS)
    results_B[label] = {'results': res, 'dim': X_combined.shape[1]}
    print_results(label, res, dim_info=f'(combined_dim={X_combined.shape[1]})')

print('=' * 70)

## 8. 결과 비교 테이블

In [ ]:
def make_summary_df(results_dict):
    rows = []
    for label, val in results_dict.items():
        clf, r = best(val['results'])
        rows.append({
            'Experiment':    label,
            'Combined Dim':  val['dim'],
            'Best Clf':      clf,
            'Soft Acc (%)':  f"{r['soft_mean']:.2f} ± {r['soft_std']:.2f}",
            'Strict Acc (%)':f"{r['strict_mean']:.2f} ± {r['strict_std']:.2f}",
            'F1 (%)':        f"{r['f1_mean']:.2f}",
            '_soft':         r['soft_mean'],
        })
    df = pd.DataFrame(rows).sort_values('_soft', ascending=False)
    return df.drop(columns=['_soft'])

print('=== 실험 A: Inter_PI + Ord_PI ===')
df_A = make_summary_df(results_A)
print(df_A.to_string(index=False))

print()
print('=== 실험 B: 3D_PI + Ord_PI ===')
df_B = make_summary_df(results_B)
print(df_B.to_string(index=False))

# 단독 기준치 (비교용)
print()
print('=== 단독 성능 기준치 ===')
for name in ['Inter_PI', 'Ord_PI', '3D_PI']:
    res = evaluate(raw[name]['X'], raw[name]['y'], pca_dim=PCA_DIM, n_splits=N_SPLITS)
    clf, r = best(res)
    print(f'  {name:10s}: Soft {r["soft_mean"]:.2f}±{r["soft_std"]:.2f}% | Best: {clf}')

## 9. 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

for ax, (results_dict, title) in zip(axes, [
    (results_A, 'Inter_PI + Ord_PI'),
    (results_B, '3D_PI + Ord_PI'),
]):
    labels = list(results_dict.keys())
    soft_means  = [best(results_dict[l]['results'])[1]['soft_mean']   for l in labels]
    soft_stds   = [best(results_dict[l]['results'])[1]['soft_std']    for l in labels]
    strict_means= [best(results_dict[l]['results'])[1]['strict_mean'] for l in labels]

    # Sort by soft accuracy
    order = sorted(range(len(labels)), key=lambda i: soft_means[i])
    labels_s  = [labels[i].split(':')[0]  for i in order]
    smeans_s  = [soft_means[i]             for i in order]
    sstds_s   = [soft_stds[i]              for i in order]
    stmeans_s = [strict_means[i]           for i in order]

    y_pos = np.arange(len(labels_s))
    ax.barh(y_pos - 0.2, smeans_s,  0.35, xerr=sstds_s, label='Soft',   alpha=0.85, capsize=3, color='#3498db')
    ax.barh(y_pos + 0.2, stmeans_s, 0.35, label='Strict', alpha=0.85, color='#e67e22')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels_s, fontsize=9)
    ax.set_xlabel('Accuracy (%)')
    ax.set_title(f'{title}\n(PCA {PCA_DIM}D, {N_SPLITS}-fold CV)', fontweight='bold')
    ax.legend()
    ax.grid(axis='x', alpha=0.3)
    ax.set_xlim([40, 100])

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/URP/phase4_combination_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

## 10. PCA 차원별 추가 실험 (Best Method)

가장 성능 좋은 결합 방법에 대해 PCA 차원을 변경하며 최적 setting 탐색.

In [ ]:
PCA_DIMS = [10, 20, 50, 100, None]   # None = No PCA (full dim)
TARGET_CLF_NAME = 'SVM (Linear, C=1)'

# Best method 자동 선택 (실험 A 기준)
best_label_A = max(results_A, key=lambda l: best(results_A[l]['results'])[1]['soft_mean'])
best_method_A = best_label_A.split(':')[0].replace('A_','').split('_')[0].lower()

print(f'Best method (A): {best_label_A}')
print()

# best method의 설정 재구성 (수동 지정 필요 시 아래 수정)
# 여기서는 df_A 1위 방법의 X를 직접 재계산
top_experiments = df_A.head(3)['Experiment'].tolist()

print('=== 상위 3개 방법 × PCA 차원별 비교 (SVM Linear) ===')
pca_scan_results = {}

for exp_label in top_experiments:
    method_info = [e for e in experiments_A if e[0] == exp_label]
    if not method_info: continue
    _, method, kwargs = method_info[0]
    X_c = combine_datasets(X_inter, X_ord, method=method, **kwargs)

    print(f'\n[{exp_label}]')
    pca_scan_results[exp_label] = {}

    for dim in PCA_DIMS:
        res = evaluate(X_c, y, pca_dim=dim, n_splits=N_SPLITS)
        if TARGET_CLF_NAME in res:
            r = res[TARGET_CLF_NAME]
            dim_label = f'{dim}D' if dim else 'NoPCA'
            pca_scan_results[exp_label][dim_label] = r
            print(f'  PCA {dim_label:6s} → Soft: {r["soft_mean"]:.2f}±{r["soft_std"]:.2f}% | Strict: {r["strict_mean"]:.2f}%')

print('\n완료')

## 11. 결과 저장

In [ ]:
save_dir = '/content/drive/MyDrive/URP/Phase4_Results'
os.makedirs(save_dir, exist_ok=True)

df_A.to_csv(os.path.join(save_dir, 'phase4_InterOrd_comparison.csv'), index=False)
df_B.to_csv(os.path.join(save_dir, 'phase4_3DOrd_comparison.csv'),   index=False)

print(f'✓ CSV 저장: {save_dir}/')
print()
print('=== 최종 요약 ===')
print()
print('Inter_PI + Ord_PI 상위 3개:')
print(df_A.head(3)[['Experiment','Soft Acc (%)','Strict Acc (%)']].to_string(index=False))
print()
print('3D_PI + Ord_PI 상위 3개:')
print(df_B.head(3)[['Experiment','Soft Acc (%)','Strict Acc (%)']].to_string(index=False))